# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import boto3
import s3fs
import json
import tarfile 
import xgboost as xgb_lib
# Weather data library
from meteostat import Stations, Daily
from datetime import datetime
from pyathena import connect

import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import seaborn as sns

import sagemaker
from sagemaker.inputs import TrainingInput
from sagemaker import image_uris
from sagemaker.serializers import CSVSerializer

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,accuracy_score, precision_score,recall_score, f1_score)
                                

# Connect SageMaker to S3

In [ ]:
#Create S3 client using boto3
#This allows the notebook to communicate with S3
s3 = boto3.client('s3')

#Define your S3 bucket name where your datasets are stored
bucket = "vegetation-risk-ml"

#list files inside bucket
response = s3.list_objects_v2(Bucket=bucket)


processed/
raw/
raw/fire/
raw/fire/California_Historic_Fire_data.csv
raw/forest/
raw/forest/CA_SUBPLOT.csv
raw/forest/CA_TREE.csv
raw/forest/CA_TREE_REGIONAL_BIOMASS.csv
raw/weather/


# Load ML dataset from S3

In [ ]:
vegetation_risk_data = pd.read_parquet("s3://vegetation-risk-ml/final/ml_ready/vegetation_ml_dataset.parquet")

print(vegetation_risk_data.shape)
vegetation_risk_data.head()

# Select features for training

In [ ]:
#Map trim_priority to numeric labels
mapping = {'Low': 0, 'Medium': 1, 'High': 2}
vegetation_risk_data['target'] = vegetation_risk_data['trim_priority'].map(mapping)
if vegetation_risk_data['target'].isnull().any():
    print("Some trim_priority values were not mapped and resulted in NaN in the target column.")
else:
    print("All trim_priority values successfully mapped to target labels.")


#Save lat/lon/county before split
meta = vegetation_risk_data[['lat', 'lon', 'countycd']].copy().reset_index(drop=True)

In [ ]:
# Select features and target variable
features=['dia', 'ht', 'slope', 'regional_drybiot', 'fire_recurrence', 'avg_temp', 'avg_rain', 'avg_wind', 'fuel_moisture_risk', 'log_fire_size','fire_month_sin', 'fire_month_cos']

X = vegetation_risk_data[features].reset_index(drop=True)
y = vegetation_risk_data["target"].reset_index(drop=True)


# Train Test Split

In [ ]:
#Use a stratified 70,15,15 (training,testing,validation) split to preserve class distribution across all three priority levels
#Split into 70% train, 30% temp (which will be further split into val and test)
X_train, X_temp, y_train, y_temp, idx_train, idx_temp = train_test_split(X, y, X.index, test_size=0.30, random_state=42, stratify=y)

# Split temp into 15% val, 15% test
X_val, X_test, y_val, y_test, idx_val, idx_test = train_test_split(X_temp, y_temp, idx_temp, test_size=0.50, random_state=42, stratify=y_temp)

# Reset indices to avoid misalignment
X_train = X_train.reset_index(drop=True)
X_val   = X_val.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_val   = y_val.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Validation shape:", X_val.shape)

# Save Splits to S3

In [ ]:
#Save and upload all three splits
def save_to_s3(X, y, bucket, key):
    df = pd.concat([y.reset_index(drop=True), X.reset_index(drop=True)], axis=1)
    df.to_csv(f's3://{bucket}/{key}', header=False, index=False)

BUCKET = 'vegetation-risk-ml'
save_to_s3(X_train, y_train, BUCKET, 'ml/train/train.csv')
save_to_s3(X_val,   y_val,   BUCKET, 'ml/val/val.csv')
save_to_s3(X_test,  y_test,  BUCKET, 'ml/test/test.csv')

# Training

This project uses the SageMaker built-in XGBoost algorithm, which provides a pre-configured container for model training without requiring custom scripts or containerization

In [ ]:
#Set up SageMaker Estimator for XGBoost
role = sagemaker.get_execution_role()
session = sagemaker.Session()

# Retrieve the XGBoost container URI
container = image_uris.retrieve('xgboost', session.boto_region_name, '1.7-1')

#Define metric definitions to capture training and validation metrics from the XGBoost logs
metric_definitions = [
    {"Name": "train:merror", "Regex": "train-merror=([0-9\\.]+)"},
    {"Name": "validation:merror", "Regex": "validation-merror=([0-9\\.]+)"},
    {"Name": "train:logloss", "Regex": "train-mlogloss=([0-9\\.]+)"},
    {"Name": "validation:logloss", "Regex": "validation-mlogloss=([0-9\\.]+)"}
]
#Create SageMaker Estimator for XGBoost
xgb = sagemaker.estimator.Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type='ml.m5.large', 
    output_path=f's3://{BUCKET}/ml/model-output/',
    sagemaker_session=session,
    metric_definitions=metric_definitions
)
#Set hyperparameters for XGBoost
xgb.set_hyperparameters(
    num_class=3,
    objective='multi:softprob',
    eval_metric= ['merror','mlogloss'],
    num_round=150,
    eta=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=1,
    reg_lambda=1,     
    reg_alpha=0.1,   
    seed=42
)
#Define training and validation data inputs for SageMaker
train_input = TrainingInput(f's3://{BUCKET}/ml/train/', content_type='text/csv')
val_input = TrainingInput(f's3://{BUCKET}/ml/val/', content_type='text/csv')

#Train the model
xgb.fit({'train': train_input, 'validation': val_input})
print('Training completed')

# Deploy

In [ ]:
#Deploy the model to an endpoint
predictor = xgb.deploy(instance_type='ml.m5.large', initial_instance_count=1)

predictor.serializer = CSVSerializer()

# Evaluation

In [ ]:
#Make predictions on the test set
raw = predictor.predict(X_test.values)

#If the output is in bytes, decode it to string
if isinstance(raw, bytes):
    raw = raw.decode()

rows = raw.strip().split('\n')
probs = np.array([list(map(float, r.split(','))) for r in rows])

#Get predicted class labels from probabilities
pred_class = np.argmax(probs, axis=1)

print(classification_report(y_test, pred_class))

print("Accuracy:", accuracy_score(y_test, pred_class))

# Feature Importance

In [ ]:
# Get model path from SageMaker
model_path = xgb.latest_training_job.describe()['ModelArtifacts']['S3ModelArtifacts']
print("Model:", model_path)

# Extract bucket & key automatically
bucket = model_path.split('/')[2]
key = '/'.join(model_path.split('/')[3:])

# Download model
boto3.client('s3').download_file(bucket, key, 'model.tar.gz')

# Extract model
tarfile.open('model.tar.gz').extractall()

# Load model correctly
model = xgb_lib.Booster()
model.load_model('xgboost-model')

# Plot feature importance
xgb_lib.plot_importance(model, max_num_features=10)
plt.title("Top Features")
plt.tight_layout()
plt.show()

# Monitor Training with CloudWatch

In [ ]:
# Retrieve training logs from CloudWatch
logs = boto3.client('logs')
log_group = '/aws/sagemaker/TrainingJobs'
job_name = xgb.latest_training_job.name

# Get latest log stream
streams = logs.describe_log_streams(
    logGroupName=log_group,
    logStreamNamePrefix=job_name,
    orderBy='LastEventTime',
    descending=True
)['logStreams']

if streams and len(streams) > 0:
    log_stream = streams[0]['logStreamName']

    next_token = None

    while True:
        kwargs = {
            'logGroupName': log_group,
            'logStreamName': log_stream,
            'startFromHead': True
        }

        if next_token:
            kwargs['nextToken'] = next_token

        response = logs.get_log_events(**kwargs)

        for event in response['events']:
            print(event['message'])

        # Stop when no new logs
        if next_token == response.get('nextForwardToken'):
            break

        next_token = response.get('nextForwardToken')

else:
    print("No log streams found yet. Training may still be starting.")

# Batch Inference

In [ ]:
# Save full data
X.to_csv(f's3://{BUCKET}/ml/batch-input/batch.csv', header=False, index=False)

#Set up batch transform job
transformer = xgb.transformer(
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=f's3://{BUCKET}/ml/batch-output/',
    accept='text/csv'
)

#Start batch transform job on the full dataset
transformer.transform(
    data=f's3://{BUCKET}/ml/batch-input/',
    content_type='text/csv',
    split_type='Line'
)
#Wait for the batch transform job to complete
transformer.wait()

# Process Batch Output

In [ ]:
#Read batch predictions from S3
batch_preds = pd.read_csv(f's3://{BUCKET}/ml/batch-output/batch.csv.out',header=None)

# Get predicted class labels and risk scores from batch predictions
probs = batch_preds.values.reshape(-1, 3)

# Predicted class is the one with highest probability
pred_class = np.argmax(probs, axis=1)
risk_score = probs[:, 2] * 100

In [ ]:
# Combine predictions with original data for analysis
results = vegetation_risk_data.copy()

# Add predictions and risk scores to results dataframe
results['predicted_class'] = pred_class
results['priority'] = pd.Series(pred_class).map({0:'Low',1:'Medium',2:'High'})
results['risk_score'] = risk_score.round(2)

# Reorder columns for better readability
results = results[features + ['lat', 'lon', 'countycd', 'priority', 'risk_score']]

# Sort results by risk score to identify highest risk areas
results = results.sort_values('risk_score', ascending=False)

print(results.head())


# Save the results to S3

In [ ]:
# Save predictions to S3
s3 = boto3.client('s3')

csv_buffer = results.to_csv(index=False)
s3.put_object(Bucket=BUCKET,Key='ml/predictions/test_predictions.csv',Body=csv_buffer)

print(" Test predictions saved")

# Delete any endpoint

In [ ]:
predictor.delete_endpoint()
predictor.delete_endpoint_config()

print("Endpoint deleted")